# 1D-CNN v5 — ECA + Mixup + Option A/B

Best 1D-CNN pipeline: raw signal, ECA attention, mixup + noise augmentation, label smoothing.

In [1]:
import sys, math
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from config import RANDOM_SEED, N_CLASSES, MODELS_DIR, get_device
from src.experiment_runner import (
    get_splits, load_and_norm, split_cal_test,
    run_zero_shot, run_calibration, print_comparison,
    TEST_SUBJECTS, TRAIN_SUBJECTS,
)
from src.evaluation import measure_latency, print_latency

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
DEVICE = get_device()
splits = get_splits()
print(f'Device: {DEVICE}')

Device: mps


## Model

In [2]:
class ECA1d(nn.Module):
    def __init__(self,ch):
        super().__init__()
        k = max(int(abs(math.log2(ch)/2+0.5)),3)
        k = k if k%2 else k+1
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.conv = nn.Conv1d(1,1,k,padding=k//2,bias=False)
    def forward(self,x):
        b,c,t = x.size()
        w = self.gap(x).view(b,1,c)
        return x * torch.sigmoid(self.conv(w)).view(b,c,1).expand_as(x)

class SepConv1d(nn.Module):
    def __init__(self,ic,oc,k=5,p=2):
        super().__init__()
        self.dw = nn.Conv1d(ic,ic,k,padding=p,groups=ic)
        self.pw = nn.Conv1d(ic,oc,1)
    def forward(self,x): return self.pw(self.dw(x))

class TemporalSCNN(nn.Module):
    def __init__(self,in_ch=8,n_classes=N_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_ch,64,5,padding=2), nn.BatchNorm1d(64), nn.ReLU(), ECA1d(64), nn.MaxPool1d(2),
            SepConv1d(64,128,5,2), nn.BatchNorm1d(128), nn.ReLU(), ECA1d(128), nn.MaxPool1d(2),
            SepConv1d(128,256,3,1), nn.BatchNorm1d(256), nn.ReLU(), ECA1d(256), nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(256,64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64,n_classes),
        )
    def forward(self,x): return self.classifier(self.features(x))
    def extract_feat(self,x):
        with torch.no_grad(): return nn.Flatten()(self.features(x))

print(f'Params: {sum(p.numel() for p in TemporalSCNN().parameters()):,}')

Params: 62,676


## Augmented dataset (mixup + noise)

In [3]:
class MixupDataset(Dataset):
    def __init__(self,X,y,alpha=0.2,noise_std=0.1):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
        self.alpha = alpha; self.noise_std = noise_std; self.n = len(y)
    def __len__(self): return self.n
    def __getitem__(self,i):
        x,y1 = self.X[i].clone(), self.y[i]
        x = x + torch.randn_like(x)*self.noise_std
        if self.alpha>0 and torch.rand(1).item()<0.5:
            j = torch.randint(0,self.n,(1,)).item()
            lam = np.random.beta(self.alpha,self.alpha)
            x = lam*x + (1-lam)*(self.X[j]+torch.randn_like(self.X[j])*self.noise_std)
            y1 = y1 if lam>=0.5 else self.y[j]
        return x, y1

@torch.no_grad()
def cnn1d_predict(X):
    model.eval()
    Xt = torch.from_numpy(X).float()
    loader = DataLoader(TensorDataset(Xt), batch_size=512, shuffle=False)
    return np.concatenate([model(xb[0].to(DEVICE)).argmax(1).cpu().numpy() for xb in loader])

@torch.no_grad()
def cnn1d_features(X):
    model.eval()
    Xt = torch.from_numpy(X).float()
    loader = DataLoader(TensorDataset(Xt), batch_size=512, shuffle=False)
    return np.concatenate([model.extract_feat(xb[0].to(DEVICE)).cpu().numpy() for xb in loader])

def cnn1d_finetune(X_cal, y_cal):
    F = cnn1d_features(X_cal)
    clf = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, C=1.0)
    clf.fit(F, y_cal)
    def predict_ft(X): return clf.predict(cnn1d_features(X))
    return predict_ft

## Train

In [4]:
train_combined = pd.concat([splits['train_df'], splits['s5_train']])
X_train, y_train, norm_stats = load_and_norm(train_combined, verbose=True)
print(f'Train: {X_train.shape}')

model = TemporalSCNN().to(DEVICE)
loader = DataLoader(MixupDataset(X_train, y_train), batch_size=256, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, epochs=50, steps_per_epoch=len(loader))
crit = nn.CrossEntropyLoss(label_smoothing=0.1)

for ep in range(50):
    model.train()
    ls,c,t = 0,0,0
    for xb,yb in loader:
        xb,yb = xb.to(DEVICE),yb.to(DEVICE)
        out = model(xb); loss = crit(out,yb)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        ls += loss.item()*xb.size(0); c += (out.argmax(1)==yb).sum().item(); t += xb.size(0)
    if (ep+1)%10==0 or ep==0:
        print(f'Epoch {ep+1:2d}/50 — loss:{ls/t:.4f} acc:{c/t:.4f}')

torch.save(model.state_dict(), MODELS_DIR / '1dcnn_v5.pt')
print('Saved.')

Loading windows: 100%|██████████| 5790/5790 [00:01<00:00, 3162.42it/s]


Train: (651972, 8, 50)
Epoch  1/50 — loss:1.7723 acc:0.3009
Epoch 10/50 — loss:1.1332 acc:0.6871
Epoch 20/50 — loss:1.0717 acc:0.7204
Epoch 30/50 — loss:1.0239 acc:0.7460
Epoch 40/50 — loss:0.9614 acc:0.7775
Epoch 50/50 — loss:0.9269 acc:0.7949
Saved.


## Option B — Zero-shot

In [5]:
print('Option B — Zero-shot:')
zero_results = run_zero_shot(cnn1d_predict, splits, norm_stats, name='1D-CNN')

Option B — Zero-shot:
  S1 zero-shot: 0.5458
  S2 zero-shot: 0.5123
  S3 zero-shot: 0.5153
  S4 zero-shot: 0.6571
  S5 zero-shot: 0.8174


## Option A — Calibration

Extract features from 1D-CNN, train LogisticRegression on calibration data.

In [6]:
print('Option A — Calibration:')
cal_results = run_calibration(cnn1d_predict, cnn1d_finetune, splits, norm_stats, name='1D-CNN')

Option A — Calibration:
  S1 calibrated: 0.7317
  S2 calibrated: 0.7718
  S3 calibrated: 0.7633
  S4 calibrated: 0.8343
  S5 calibrated: 0.8699


## Latency

In [7]:
model.eval()
s = torch.randn(1,8,50).to(DEVICE)
for _ in range(10): _ = model(s)
if DEVICE.type=='mps': torch.mps.synchronize()
def cnn1d_single(x):
    xt = torch.from_numpy(x).float().to(DEVICE)
    with torch.no_grad(): out = model(xt)
    if DEVICE.type=='mps': torch.mps.synchronize()
    return out.argmax(1).cpu().numpy()
latency = measure_latency(cnn1d_single, X_train[:1], n_runs=500)
print_latency(latency, '1D-CNN')


Latency — 1D-CNN
  Mean:   3.39 ms
  Median: 3.09 ms
  P95:    6.54 ms
  <300ms: ✓


## Results

In [8]:
print_comparison(zero_results, cal_results, name='1D-CNN (v5)')


  1D-CNN (v5) — RESULTS
Scenario        Zero-shot   Calibrated        Δ
-------------------------------------------------------
S1                54.58%       73.17%   +18.60%
S2                51.23%       77.18%   +25.96%
S3                51.53%       76.33%   +24.80%
S4                65.71%       83.43%   +17.72%
S5                81.74%       86.99% ✓   +5.25%
